In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib

Using matplotlib backend: module://matplotlib_inline.backend_inline


In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)
print(stoi)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}


In [6]:
# Build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X,Y = [], []
for w in words[:5]:
    
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join([itos[i] for i in context]), '--->', itos[ix])

        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [7]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [10]:
C = torch.randn((27,2))
C

tensor([[-4.7201e-01,  4.4918e-01],
        [ 1.4536e-01,  6.8440e-01],
        [ 1.4726e+00,  2.4275e+00],
        [-2.0769e-01, -4.5712e-01],
        [-1.9761e+00, -1.0367e+00],
        [-3.0132e-01, -2.4224e-03],
        [-1.8920e-01,  4.5834e-01],
        [-4.3786e-01,  7.5971e-01],
        [ 1.5727e+00, -1.2696e+00],
        [ 1.3366e-01,  1.7706e+00],
        [ 6.7483e-02, -7.4759e-01],
        [ 4.2572e-01,  9.3794e-01],
        [ 1.3324e+00,  1.1944e+00],
        [ 1.8300e+00, -9.6363e-01],
        [-3.6436e-01, -6.2941e-01],
        [-1.0673e+00, -9.8326e-01],
        [-2.0013e-01, -8.7178e-01],
        [ 5.7892e-01,  6.1324e-01],
        [-9.3394e-01,  3.2632e-01],
        [-1.2221e+00, -1.1874e-02],
        [-1.2750e+00, -4.3242e-01],
        [ 5.7940e-01, -5.0007e-01],
        [ 3.1817e-01, -2.8949e+00],
        [-1.1247e+00, -7.3914e-01],
        [ 1.5966e+00, -8.7489e-01],
        [-1.5470e-01, -1.3834e+00],
        [-9.1446e-01, -9.8832e-01]])

In [30]:
# We can index into tensors with multidimensional tensors

print("X[3]",X[3])
print("X[3] to letters", [itos[i.item()] for i in X[3]])
print("X[3] to embeddings using list comprehension", [C[i] for i in X[3]])
print("X[3] to embeddings using indexing", C[X[3]])

X[3] tensor([ 5, 13, 13])
X[3] to letters ['e', 'm', 'm']
X[3] to embeddings using list comprehension [tensor([-0.3013, -0.0024]), tensor([ 1.8300, -0.9636]), tensor([ 1.8300, -0.9636])]
X[3] to embeddings using indexing tensor([[-0.3013, -0.0024],
        [ 1.8300, -0.9636],
        [ 1.8300, -0.9636]])


In [ ]:
emb = C[X]
emb.shape
# 32 examples with 3 letters with a 2 dim embedding per word

torch.Size([32, 3, 2])

In [ ]:
torch.cat(torch.unbind(emb, 1), 1).shape # removes a dimension then concats them to get the desired shape
                                         # Inefficient because it is creating an entirely new tensor

torch.Size([32, 6])

In [ ]:
W1 = torch.randn(())